**Context-Aware Chatbot Using LangChain + RAG**

**Problem Statement**

Build a context-aware chatbot using retrieval-based techniques.

 **Methodology**

Used LangChain, embeddings, and FAISS to retrieve relevant information and generate responses.

**Install Libraries**

Install required libraries for building a chatbot with retrieval capabilities.

In [3]:
!pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


**Import Libraries**

Import necessary modules for document processing, embeddings, and LLM integration.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from transformers import pipeline

**Create Knowledge Base**

Create a custom text dataset that will act as the chatbot’s knowledge source.

In [6]:
text = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning is a subset of machine learning that uses neural networks.
AI is widely used in healthcare, education, and business applications.
"""

with open("knowledge.txt", "w") as f:
    f.write(text)

**Load Documents**

Load the text data into the system for processing.

In [7]:
loader = TextLoader("knowledge.txt")
documents = loader.load()

**Split Text into Chunks**

Split the document into smaller chunks for efficient retrieval.

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.split_documents(documents)

**Create Embeddings**

Convert text data into numerical vector representations.

In [9]:
embeddings = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Create Vector Store (FAISS)**

Store text embeddings in a vector database for similarity search.

In [10]:
db = FAISS.from_documents(docs, embeddings)

**Load Language Model**

Load a pretrained language model for generating responses.

In [11]:
pipe = pipeline("text-generation", model="gpt2")
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/tmp/ipykernel_7094/436415406.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


**Create Retrieval-Based QA System**

Combine the language model with vector search to answer questions from knowledge base.

In [12]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever()
)

**Ask a Question**

Query the chatbot to generate an answer based on stored knowledge.

In [13]:
query = "What is Artificial Intelligence?"
result = qa.run(query)
print(result)

/tmp/ipykernel_7094/3704618464.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa.run(query)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning is a subset of machine learning that uses neural networks.
AI is widely used in healthcare, education, and business applications.

Question: What is Artificial Intelligence?
Helpful Answer: Artificial Intelligence is a subset of human intelligence, consisting of two types:

Intelligent

Machine-learning

Machine-learning machines are thought to be better at solving problems than humans.

There are some things about human intelligence that we can learn from, such as the ability to read, write, and perform math, and the ability to think. Machine-learning machines can also help us to understand and solve questions.

Machine learning machines are machines that 

**Interactive Chat**

Create an interactive loop to allow continuous conversation with the chatbot.

In [16]:
while True:
    q = input("Ask something (type 'exit' to stop): ")
    if q.lower() == "exit":
        break
    print("Answer:", qa.run(q))

Ask something (type 'exit' to stop): What is deep learning?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning is a subset of machine learning that uses neural networks.
AI is widely used in healthcare, education, and business applications.

Question: What is deep learning?
Helpful Answer: Deep learning is a form of artificial intelligence that allows a human to learn something from a wide range of data and, in a very few cases, even to perform more complex tasks. In a few cases, the algorithms can learn to solve complex problems or to solve a few complex problems simultaneously. Deep learning algorithms can also be used to analyze computer programs.

The human brain is a very large neural network. The human brain is not a single piece of software. The human

**Results**

The chatbot successfully answered queries using the provided knowledge base.

**Conclusion**

The chatbot demonstrates how combining retrieval systems with language models improves accuracy and enables context-aware responses.